<a href="https://colab.research.google.com/github/RautRitesh/langgraph/blob/main/langchain_pageindex_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![pageindex_banner](https://pageindex.ai/static/images/pageindex_banner.jpg)

# Agentic RAG with PageIndex MCP + LangChain

[PageIndex](https://github.com/VectifyAI/PageIndex) is a vectorless, reasoning-based retrieval framework. It transforms documents into hierarchical tree indexes and uses LLM reasoning to navigate the structure — without chunking, embeddings, or vector databases.

[PageIndex MCP](https://pageindex.ai/mcp) exposes this capability as a set of MCP tools (`get_document_structure`, `get_page_content`, etc.), enabling any MCP-compatible agent to retrieve from your documents.

This notebook demonstrates **agentic RAG** using [LangChain](https://github.com/langchain-ai/langchain) with [langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters) — the official bridge that converts MCP tools into LangChain-compatible tools, so you can use PageIndex MCP inside any LangGraph agent.

### Install

##small optimization.

In [ ]:
%pip install -q --upgrade langchain-mcp-adapters  pageindex langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00


### Setup

In [ ]:
from google.colab import userdata
api_key=userdata.get('groq_api_key_2')
page_index_api_key=userdata.get('page_index_api_key')


In [ ]:
system_prompt_text="""
You are an advanced legal AI assistant.
To answer the user's question, you MUST use the provided tools to extract information from the documents.
When using a tool, provide ONLY the required arguments and strictly follow the tool's parameter schema. Do not add conversational filler before calling a tool.
"""

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from pageindex import PageIndexClient
from langchain_core.messages import SystemMessage

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = page_index_api_key

# Get your Anthropic API key from https://platform.claude.com/settings/keys

MODEL = "openai/gpt-oss-120b"

pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)
llm = ChatGroq(
    model=MODEL,
    api_key=api_key,
)
mcp_client = MultiServerMCPClient({"pageindex": {"transport": "http", "url": "https://api.pageindex.ai/mcp", "headers": {"Authorization": f"Bearer {PAGEINDEX_API_KEY}"}}})


async def ask(question, show_tool_results=False):
    tools = await mcp_client.get_tools()
    agent = create_agent(llm, tools,system_prompt=SystemMessage(content=system_prompt_text))
    current = None
    async for event in agent.astream_events({"messages": question}, version="v2"):
        kind = event["event"]
        data = event["data"]
        if kind == "on_chat_model_stream":
            for block in (data["chunk"].content if isinstance(data["chunk"].content, list) else []):
                btype = block.get("type")
                if btype != current:
                    if btype == "thinking":
                        print("\n[thinking] ", end="", flush=True)
                    elif btype == "text":
                        print("\n", end="", flush=True)
                    current = btype
                if btype == "thinking":
                    print(block.get("thinking", ""), end="", flush=True)
                elif btype == "text":
                    print(block.get("text", ""), end="", flush=True)
        elif kind == "on_tool_start":
            current = None
            print(f"\n[tool_use] {event['name']} {data['input']}")
        elif kind == "on_tool_end" and show_tool_results:
            print(f"\n[tool_result] {data['output']}")

### Upload a PDF

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from pageindex import PageIndexClient
from langchain_core.messages import SystemMessage

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = page_index_api_key

# Setting the model (You can change this to "llama3-groq-70b-8192-tool-use-preview" if the versatile one still fails)
MODEL = "groq/compound"

pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)
llm = ChatGroq(
    model=MODEL,
    api_key=api_key,
)
mcp_client = MultiServerMCPClient({"pageindex": {"transport": "http", "url": "https://api.pageindex.ai/mcp", "headers": {"Authorization": f"Bearer {PAGEINDEX_API_KEY}"}}})

async def ask(question):
    tools = await mcp_client.get_tools()

    # 1. Stronger system prompt to enforce strict tool usage
    system_prompt_text = """
    You are an advanced legal AI assistant.
    To answer the user's question, you MUST use the provided tools to extract information from the documents.
    When using a tool, provide ONLY the required arguments and strictly follow the tool's parameter schema.
    Do not add conversational filler before calling a tool.
    """

    agent = create_agent(llm, tools, system_prompt=SystemMessage(content=system_prompt_text))

    print(f"Asking: '{question}'\n")
    print("Agent is thinking and searching the document... please wait.\n")

    try:
        # 2. Use ainvoke to wait for the final complete response instead of parsing a stream
        result = await agent.ainvoke({"messages": question})

        # The agent returns a dictionary where "messages" contains the conversation history
        # The last message is the final answer generated by the LLM
        final_answer = result["messages"][-1].content

        print("### Final Answer ###")
        print(final_answer)

    except Exception as e:
        print(f"\n[!] An error occurred during execution: {e}")
        print("\nTip: If you see 'APIError: Failed to call a function', try changing the MODEL variable to 'llama3-groq-70b-8192-tool-use-preview'.")

In [ ]:
import os, requests, time

pdf_path = "/content/najir_compressed.pdf"

doc_id = pi.submit_document(pdf_path)["doc_id"]
print(f"Submitted: {doc_id}")

# Wait for processing
while pi.get_document(doc_id)["status"] != "completed":
    time.sleep(5)
print(f"Ready: {pi.get_document(doc_id)['name']}")

Submitted: pi-cmp8nb8i7001801qz1hihft8a
Ready: najir_compressed.pdf


### Ask a question

LangChain connects to PageIndex MCP via `langchain-mcp-adapters`, converts MCP tools to LangChain tools, and runs a ReAct agent that autonomously calls tools and reasons over the results.

In [ ]:
await ask("Use the tool for my question. What are my legal rights as a loan guarantor if a bank or finance company auctions my collateral land to recover a debt without giving me prior notice?")

Asking: 'Use the tool for my question. What are my legal rights as a loan guarantor if a bank or finance company auctions my collateral land to recover a debt without giving me prior notice?'

Agent is thinking and searching the document... please wait.


[!] An error occurred during execution: Error code: 400 - {'error': {'message': '`tool calling` is not supported with this model', 'type': 'invalid_request_error', 'param': 'tool calling'}}

Tip: If you see 'APIError: Failed to call a function', try changing the MODEL variable to 'llama3-groq-70b-8192-tool-use-preview'.
